June 20, 2026 

COURT CASES

Visit https://www.tnwb.uscourts.gov/Search/Search.aspxLinks to an external site. and search for "CAR." Scrape the results into a CSV, with four columns: the URL to the case, the name of the case, the category (e.g. "Judge's Opinions), the additional details (terms match/size/pdf URL).

Bonuses, if you want to get fancy:

Split up the additional details into multiple columns
Download all of the PDFs of the cases
Tips:

There are only 10 results, and so many pages! ...maybe there's a secret way to get them all on one page?

Check the class you're using and see if it matches the number of results (it probably doesn't!). Inspect the page to find out why. You have two options: use something like we did in class with item.select("h1, h2") – but slightly different, since we're talking about classes – or have two separate loops.
.split is often a convenient way to separate semi-structured text

Downloading PDFs in Python probably does not involve wget (unless you really want to)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
response = requests.get('https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_sort=0&zoom_xml=0&zoom_query=car&zoom_per_page=10&zoom_and=1&zoom_cat%5B%5D=-1')
doc = BeautifulSoup(response.text, 'lxml')   # was 'html.parser'
print(response.status_code)   # 200 means OK


200


In [3]:
cases = doc.find_all('div', class_=['result_block', 'result_altblock'])

#this is odd rows + even rows - class per row

In [4]:
# #can also be written as 

# select('.result_block, .result_altblock') 


In [5]:
len(cases)

10

In [1]:
rows = []
for case in cases:
    row = {} #this is the nested structure per case 
    
    try:
        row['name'] = case.find('a').text.strip()
    except:
        row['name'] = None

    try:
        row['category'] = case.find(class_ = 'category').text.strip()
    except:
        row['category'] = None      


    # PDF Link
    infoline = case.find(class_='infoline').text

    try:
        row['terms_matched'] = infoline.split('-')[0].split(':')[-1].strip()   # '1'
    except:
        row['terms_matched'] = None

    try:
        row['size'] = infoline.split('-')[1].strip()                  # '102k'
    except:
        row['size'] = None

    try:
        row['pdf_link'] = infoline.split('URL:')[-1].strip()          # the clean URL
    except:
        row['pdf_link'] = None

    try:
        row['context'] = case.find(class_ = 'context').text.strip()
    except:
        row['context'] = None



    rows.append(row)

len(rows)



NameError: name 'cases' is not defined

In [8]:
#how to get url and text seperately - when they are inside the div element's content 
# <div class="infoline">Terms matched:  1&nbsp; 
# - &nbsp;102k&nbsp; 
# - &nbsp;URL: https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf
# </div>


#vs
## <a href = "http://xxxx.com" > <---     
#the URL inside the <a> tag
# row['link'] = story.find('a')['href']

# <a href="https://...pdf">JDL: 04-24318 Jacquelline D. Black</a>
#    │    └──── value ────┘  └────────── text content ────────┘
#    └─ attribute name

# href = the attribute name
# "https://...pdf" = the attribute value ← grab this





Notes on .split('URL')

.split() chops a string into pieces. Cut at 'URL:':

infoline.split('URL:')
This returns a list of two pieces — everything before URL:, and everything after:

['Terms matched:  1\xa0 - \xa0102k\xa0 - \xa0',  '  https://www.tnwb.uscourts.gov/...pdf']

piece [0]  — the stuff you don't want         
piece [1] — the part with the link





In [9]:
rows

[{'name': 'JDL: 04-24318 Jacquelline D. Black',
  'category': "[Judges' Opinions]",
  'terms_matched': '1',
  'size': '102k',
  'pdf_link': 'https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf',
  'context': "... the basis that the Debtor failed to prove that K's Auto had custody of the car or knew of the whereabouts of the car. This adversary proceeding was administratively ..."},
 {'name': 'WHB: 95-26401 Mary Lucy Cooper',
  'category': "[Judges' Opinions]",
  'terms_matched': '1',
  'size': '27k',
  'pdf_link': 'https://www.tnwb.uscourts.gov/Opinions/whb/pdf/whb19950809xn1.pdf',
  'context': '... MARY LUCY COOPER, Plaintiff, v. Adversary Proceeding NO. 95-0757 ROGERS USED CARS, Defendant. ____ OPINION AND ORDER DENYING TURNOVER ____ This adversary ...'},
 {'name': 'GHB: 97-12368 Billy G. Woffard',
  'category': "[Judges' Opinions]",
  'terms_matched': '1',
  'size': '71k',
  'pdf_link': 'https://www.tnwb.uscourts.gov/Opinions/ghb/pdf/ghb19980812xn1.pdf',
  'context': '

## Part 2
#### Q1 download pdf using panda or just dictionaries 

In [10]:
df = pd.DataFrame(rows)
df.head()

,name,category,terms_matched,size,pdf_link,context
0,JDL: 04-24318 Jacquelline D. Black,[Judges' Opinions],1,102k,https://www.tnwb.uscourts.gov/Opinions/jdl/pdf...,... the basis that the Debtor failed to prove ...
1,WHB: 95-26401 Mary Lucy Cooper,[Judges' Opinions],1,27k,https://www.tnwb.uscourts.gov/Opinions/whb/pdf...,"... MARY LUCY COOPER, Plaintiff, v. Adversary ..."
2,GHB: 97-12368 Billy G. Woffard,[Judges' Opinions],1,71k,https://www.tnwb.uscourts.gov/Opinions/ghb/pdf...,"... G. Woffard, ("" Woffard"") , was partners in..."
3,JDL: 97-30580 Mary Chrlis Hurst,[Judges' Opinions],1,32k,https://www.tnwb.uscourts.gov/Opinions/jdl/pdf...,... UNITED STATES BANKRUPTCY COURT WESTERN DIS...
4,MRH: 20-20967 Jacob Braxton Herring 20-00094,[Judges' Opinions],1,303k,https://www.tnwb.uscourts.gov/Opinions/mrh/pdf...,... and soon thereafter the contract was assig...


In [11]:
df.to_csv('courtcases10.csv', index=False)

Workflow
- save as a csv
- create a folder to download all pdfs


In [15]:
import os
os.makedirs('pdf_courtcases10', exist_ok=True)   # make the folder; don't error if it exists


**** use .split to give pdf file a name first before downloading

In [17]:
# for row in rows:
# walking through rows (your list of dicts), not the DataFrame

for row in rows:
    url = row['pdf_link']
    if not url:
        continue

    # make a filename from the end of the URL: ".../jdl20071024nn1.pdf"
    filename = url.split('/')[-1]
    path = os.path.join('pdf_courtcases10', filename)

    response = requests.get(url)          # fetch the PDF
    with open(path, 'wb') as f:           # 'wb' = write BINARY (python syntax)<-- internal python work
        f.write(response.content)         # .content = raw bytes, NOT .text

    print(f'saved {filename}')


saved jdl20071024nn1.pdf
saved whb19950809xn1.pdf
saved ghb19980812xn1.pdf
saved jdl19970918pn1.pdf
saved mrh20220615nn1.pdf
saved ghb19970417pn1.pdf
saved jdl20090325nn1.pdf
saved ghb20040325nn1.pdf
saved ghb20000707nn1.pdf
saved ghb20040503nn1.pdf


#### Q2 multiple pages scrape

https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_sort=0&zoom_xml=0&zoom_query=car&zoom_per_page=10&zoom_and=1&zoom_cat%5B%5D=-1

#### Only using patterns from page 2,3,4,below instead
https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=2&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0

https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=3&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0

https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=4&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0


In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

urls = [
    'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=2&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0',
    'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=3&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0',
    'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=4&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0',
]

rows = []


In [29]:
for url in urls:
    response = requests.get(url)
    doc = BeautifulSoup(response.text, 'lxml')
    #print(url, response.status_code)


    cases = doc.find_all('div', class_=['result_block', 'result_altblock'])

    for case in cases:          # <-- inner loop
       
        row = {}
        
        try:
            row['name'] = case.find('a').text.strip()
        except:
            row['name'] = None

        try:
            row['category'] = case.find(class_ = 'category').text.strip()
        except:
            row['category'] = None      


        # PDF Link
        infoline = case.find(class_='infoline').text

        try:
            row['terms_matched'] = infoline.split('-')[0].split(':')[-1].strip()   # '1'
        except:
            row['terms_matched'] = None

        try:
            row['size'] = infoline.split('-')[1].strip()                  # '102k'
        except:
            row['size'] = None

        try:
            row['pdf_link'] = infoline.split('URL:')[-1].strip()          # the clean URL
        except:
            row['pdf_link'] = None

        try:
            row['context'] = case.find(class_ = 'context').text.strip()
        except:
            row['context'] = None

        rows.append(row)


len(rows)


30

In [31]:
df2 = pd.DataFrame(rows)
df2.to_csv('courtcases_pages234.csv', index=False)
df2.head()

,name,category,terms_matched,size,pdf_link,context
0,GHB: 02-12407 Luis Rossi and Evelyn Rossi,[Judges' Opinions],1,277k,https://www.tnwb.uscourts.gov/Opinions/ghb/pdf...,... Rossi did not inform the credit union of t...
1,GHB: 99-12067 James Dean and Patsy Dean,[Judges' Opinions],1,281k,https://www.tnwb.uscourts.gov/Opinions/ghb/pdf...,"... . In their petition, the Debtors listed a ..."
2,WHB: 95-29798 Byron Crumb,[Judges' Opinions],1,23k,https://www.tnwb.uscourts.gov/Opinions/whb/pdf...,"... Debtor Chapter 13 BYRON CRUMB, Plaintiff, ..."
3,JDL: 04-23035 Jennifer Ann Jamison-McGee,[Judges' Opinions],1,90k,https://www.tnwb.uscourts.gov/Opinions/jdl/pdf...,... application is incorrect. Although she is ...
4,JDL: 04-23035 Jennifer Ann Jamison-McGee,[Judges' Opinions],1,134k,https://www.tnwb.uscourts.gov/Opinions/jdl/pdf...,... application is incorrect. Although she is ...


### Building URLS, scaling to page 100 or 200, without manual URL entry

In [ ]:
#Step 1 - build URLs using F string {} <-- becomes the variable 
page = 2
url = f'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page={page}&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0'


In [ ]:
#url

'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=2&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0'

Multiple URLS

In [40]:
urls = []
for page in range(2, 5):          # 2, 3, 4
    url = f'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page={page}&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0'
    urls.append(url)



In [ ]:
#urls

['https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=2&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0',
 'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=3&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0',
 'https://www.tnwb.uscourts.gov/Search/search.aspx?zoom_query=car&zoom_page=4&zoom_per_page=10&zoom_and=1&zoom_sort=0&zoom_xml=0']